# Real-World Data Analysis: *Phyllanthus Niruri* and Kidney Stone Treatments

This notebook implements a streamlined analysis of kidney stone treatment effectiveness and adverse effects across three platforms: WebMD, Amazon, and Reddit.

In [1]:
# Import necessary libraries
from __future__ import annotations

import math
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats.contingency import expected_freq
from statsmodels.formula.api import glm
import statsmodels.api as sm
from statsmodels.stats.contingency_tables import Table2x2
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportion_confint

In [2]:
# ---------------------------------------------------------------------
# Constants & I/O configuration
# ---------------------------------------------------------------------
CSV_DIR = Path("csv-files")  # adjust to your local layout
FILE_MAP = {
    "WebMD": CSV_DIR / "Kidney Stone Reviews - Reviews - WebMD.csv",
    "Amazon": CSV_DIR / "Kidney Stone Reviews - Reviews - Amazon.csv",
    "Reddit": CSV_DIR / "Kidney Stone Reviews - Reviews - Reddit.csv",
}

HELP_COL = "Helps overall with kidney stones"
AE_COL = "Side effects mentioned"
HQ_COL = "Super high quality"

# methodological thresholds
OBS_MIN = 5      # descriptive vs inferential gate
EPV_MIN = 10     # events‑per‑arm threshold for logistic regression
# -----------------------------

In [3]:
# ---------------------------------------------------------------------
# Data loading & minimal cleaning
# ---------------------------------------------------------------------

def _clean_help_col(series: pd.Series) -> pd.Series:
    """Convert various encodings to {1, -1, NA}."""
    mapping = {True: 1, False: -1}
    return (
        series.replace(mapping)
        .where(series.isin([1, -1]))
        .astype("Int64")
    )

def load_kidney_stone_data() -> Dict[str, pd.DataFrame]:
    """Read each platform CSV and reproduce the notebook’s weighting tweaks."""
    data: Dict[str, pd.DataFrame] = {}
    for platform, path in FILE_MAP.items():
        df = pd.read_csv(path)

        # --- column normalisation ------------------------------------
        df[HELP_COL] = _clean_help_col(df[HELP_COL])
        df[AE_COL] = df[AE_COL].apply(lambda x: 1 if isinstance(x, str) and x.strip() else 0)

        # --- default weight + Amazon corrections for 1-star reviews being overrepresented ---
        df["Weight"] = 1.0
        if platform == "Amazon":
            chanca_piedra_corrections = {
                "NaturalisimoLife Chanca Piedra 1600 mg": {
                    "scraped": [82, 18, 37, 103, 146],
                    "actual":  [82, 18, 38, 103, 906],
                },
                "EU Natural: \"Stone Breaker\" chanca piedra": {
                    "scraped": [102, 44, 50, 100, 138],
                    "actual":  [123, 44, 50, 131, 918],
                },
            }
            for brand, corr in chanca_piedra_corrections.items():
                for i in range(5):
                    star = i + 1
                    scraped, actual = corr["scraped"][i], corr["actual"][i]
                    if scraped > 0:
                        wt = actual / scraped
                        mask = (
                            (df["Medicine"] == "Chanca piedra")
                            & (df["Source"] == brand)
                            & (df["Stars"] == star)
                        )
                        df.loc[mask, "Weight"] = wt
        data[platform] = df
    return data

In [4]:
# ---------------------------------------------------------------------
# 2×2 helper and method selection
# ---------------------------------------------------------------------

def _2x2(df, ref, comp, col):
    tbl = pd.crosstab(
        df["Medicine"].map({ref: "ref", comp: "comp"}),
        df[col].eq(1).fillna(False),
        dropna=False,
    )
    tbl = tbl.reindex(index=["ref", "comp"], columns=[False, True],
                      fill_value=0)
    return tbl


def choose_method(tbl: pd.DataFrame) -> str:
    if tbl.min().min() < OBS_MIN:
        return "describe"
    if expected_freq(tbl.to_numpy()).min() < OBS_MIN:
        return "fisher"
    events, non_events = tbl[True].min(), tbl[False].min()
    return "logit" if min(events, non_events) >= EPV_MIN else "fisher"

## 1. Consolidated Summary Table

We'll create a comprehensive table showing effectiveness and adverse events across all platforms for both all reviews and high-quality reviews.

In [5]:
def create_effectiveness_table():
    """
    Create a table focused on effectiveness metrics across platforms:
    - Platform (WebMD, Amazon, Reddit)
    - Medicine name
    - Sample sizes (All / HQ)
    - Effectiveness percentages (All / HQ)
    """
    data = load_kidney_stone_data()
    
    # Products to exclude from effectiveness table (only for WebMD platform)
    webmd_excluded_products = ["Garcinia", "Black seed", "Ashwagandha", "Melatonin"]
    
    # Prepare an empty list to store effectiveness records
    effectiveness_summaries = []
    
    # Process each platform
    for platform, df in data.items():
        # Get all products with at least 5 reviews
        all_products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= 5].index.tolist()
        
        # Apply exclusion only for WebMD platform
        if platform == "WebMD":
            all_products = [product for product in all_products if product not in webmd_excluded_products]
        
        for product in all_products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Initialize variables
            helped_pct_all = 0
            helped_ci_low_all = float('nan')
            helped_ci_high_all = float('nan')
            n_hq = 0
            helped_pct_hq = float('nan')
            helped_ci_low_hq = float('nan')
            helped_ci_high_hq = float('nan')
            
            # Calculate effectiveness using weights
            helped_mask = all_df[HELP_COL].eq(1).fillna(False)
            weighted_helped = all_df.loc[helped_mask, 'Weight'].sum() if any(helped_mask) else 0
            total_weight = all_df['Weight'].sum()
            helped_pct_all = 100 * weighted_helped / total_weight if total_weight > 0 else 0
            
            # Add Wilson CIs
            if n_all > 0:
                # Effective counts for CI calculation
                effective_count = weighted_helped / total_weight * n_all if total_weight > 0 else 0
                if effective_count > 0:
                    helped_ci_low_all, helped_ci_high_all = proportion_confint(
                        effective_count, n_all, method="wilson"
                    )
                    helped_ci_low_all *= 100
                    helped_ci_high_all *= 100
            
            # HIGH QUALITY reviews (if applicable)
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq > 0:
                    # Calculate effectiveness for HQ
                    helped_mask_hq = hq_df[HELP_COL].eq(1).fillna(False)
                    helped_weighted_hq = hq_df.loc[helped_mask_hq, 'Weight'].sum() if any(helped_mask_hq) else 0
                    total_weight_hq = hq_df['Weight'].sum()
                    helped_pct_hq = 100 * helped_weighted_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # For CI calculation, use effective counts
                    if n_hq > 0:
                        effective_helped_hq = helped_weighted_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_helped_hq > 0:
                            helped_ci_low_hq, helped_ci_high_hq = proportion_confint(
                                effective_helped_hq, n_hq, method="wilson"
                            )
                            helped_ci_low_hq *= 100
                            helped_ci_high_hq *= 100
            
            # Add record to summaries
            effectiveness_summaries.append({
                "Platform": platform,
                "Medicine": product,
                "N_All": n_all,
                "N_HQ": n_hq,
                "Helped_Pct_All": helped_pct_all,
                "Helped_CI_Low_All": helped_ci_low_all,
                "Helped_CI_High_All": helped_ci_high_all,
                "Helped_Pct_HQ": helped_pct_hq,
                "Helped_CI_Low_HQ": helped_ci_low_hq,
                "Helped_CI_High_HQ": helped_ci_high_hq
            })
    
    # Convert to DataFrame and format
    effectiveness_df = pd.DataFrame(effectiveness_summaries)
    
    # Create a more readable format for effectiveness with CIs
    effectiveness_df["Effectiveness_All"] = effectiveness_df.apply(
        lambda x: f"{x['Helped_Pct_All']:.1f}% ({x['Helped_CI_Low_All']:.1f}-{x['Helped_CI_High_All']:.1f})" 
        if not pd.isna(x['Helped_CI_Low_All']) else f"{x['Helped_Pct_All']:.1f}%", 
        axis=1
    )
    
    effectiveness_df["Effectiveness_HQ"] = effectiveness_df.apply(
        lambda x: f"{x['Helped_Pct_HQ']:.1f}% ({x['Helped_CI_Low_HQ']:.1f}-{x['Helped_CI_High_HQ']:.1f})" 
        if not pd.isna(x['Helped_CI_Low_HQ']) else f"{x['Helped_Pct_HQ']:.1f}%" if not pd.isna(x['Helped_Pct_HQ']) else "N/A", 
        axis=1
    )
    
    # Select and order columns for final display
    display_cols = [
        "Platform", "Medicine", 
        "N_All", "Effectiveness_All",
        "N_HQ", "Effectiveness_HQ"
    ]
    
    # Sort by platform and then by medicine
    effectiveness_df = effectiveness_df.sort_values(["Platform", "Medicine"])
    
    return effectiveness_df[display_cols]

def create_adverse_events_table():
    """
    Create a table focused on adverse events metrics across platforms:
    - Platform (WebMD, Amazon, Reddit)
    - Medicine name
    - Sample sizes (All / HQ)
    - Adverse Events percentages (All / HQ)
    """
    data = load_kidney_stone_data()
    
    # Prepare an empty list to store adverse events records
    ae_summaries = []
    
    # Process each platform
    for platform, df in data.items():
        # Get all products with at least 5 reviews
        all_products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= 5].index.tolist()
        
        for product in all_products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Initialize variables
            ae_pct_all = 0
            ae_ci_low_all = float('nan')
            ae_ci_high_all = float('nan')
            n_hq = 0
            ae_pct_hq = float('nan')
            ae_ci_low_hq = float('nan')
            ae_ci_high_hq = float('nan')
            
            # Calculate adverse events
            ae_mask = all_df[AE_COL].eq(1).fillna(False)
            ae_weighted = all_df.loc[ae_mask, 'Weight'].sum() if any(ae_mask) else 0
            total_weight = all_df['Weight'].sum()
            
            # Calculate percentage using weights
            ae_pct_all = 100 * ae_weighted / total_weight if total_weight > 0 else 0
            
            # For CI calculation, use effective counts
            if n_all > 0:
                effective_ae = ae_weighted / total_weight * n_all if total_weight > 0 else 0
                if effective_ae > 0:
                    ae_ci_low_all, ae_ci_high_all = proportion_confint(
                        effective_ae, n_all, method="wilson"
                    )
                    ae_ci_low_all *= 100
                    ae_ci_high_all *= 100
            
            # HIGH QUALITY reviews (if applicable)
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq > 0:
                    # Calculate adverse events for HQ
                    ae_mask_hq = hq_df[AE_COL].eq(1).fillna(False)
                    ae_weighted_hq = hq_df.loc[ae_mask_hq, 'Weight'].sum() if any(ae_mask_hq) else 0
                    total_weight_hq = hq_df['Weight'].sum()
                    ae_pct_hq = 100 * ae_weighted_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # For CI calculation, use effective counts
                    if n_hq > 0:
                        effective_ae_hq = ae_weighted_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_ae_hq > 0:
                            ae_ci_low_hq, ae_ci_high_hq = proportion_confint(
                                effective_ae_hq, n_hq, method="wilson"
                            )
                            ae_ci_low_hq *= 100
                            ae_ci_high_hq *= 100
            
            # Add record to summaries
            ae_summaries.append({
                "Platform": platform,
                "Medicine": product,
                "N_All": n_all,
                "N_HQ": n_hq,
                "AE_Pct_All": ae_pct_all,
                "AE_CI_Low_All": ae_ci_low_all,
                "AE_CI_High_All": ae_ci_high_all,
                "AE_Pct_HQ": ae_pct_hq,
                "AE_CI_Low_HQ": ae_ci_low_hq,
                "AE_CI_High_HQ": ae_ci_high_hq
            })
    
    # Convert to DataFrame and format
    ae_df = pd.DataFrame(ae_summaries)
    
    # Create a more readable format for AE with CIs
    ae_df["Adverse_Events_All"] = ae_df.apply(
        lambda x: f"{x['AE_Pct_All']:.1f}% ({x['AE_CI_Low_All']:.1f}-{x['AE_CI_High_All']:.1f})" 
        if not pd.isna(x['AE_CI_Low_All']) else f"{x['AE_Pct_All']:.1f}%", 
        axis=1
    )
    
    ae_df["Adverse_Events_HQ"] = ae_df.apply(
        lambda x: f"{x['AE_Pct_HQ']:.1f}% ({x['AE_CI_Low_HQ']:.1f}-{x['AE_CI_High_HQ']:.1f})" 
        if not pd.isna(x['AE_CI_Low_HQ']) else f"{x['AE_Pct_HQ']:.1f}%" if not pd.isna(x['AE_Pct_HQ']) else "N/A", 
        axis=1
    )
    
    # Select and order columns for final display
    display_cols = [
        "Platform", "Medicine", 
        "N_All", "Adverse_Events_All",
        "N_HQ", "Adverse_Events_HQ"
    ]
    
    # Sort by platform and then by medicine
    ae_df = ae_df.sort_values(["Platform", "Medicine"])
    
    return ae_df[display_cols]

# Function to run both tables and save them
def create_and_save_split_tables():
    """
    Create both effectiveness and adverse events tables and save them to CSV
    """
    # Generate the tables
    effectiveness_table = create_effectiveness_table()
    adverse_events_table = create_adverse_events_table()
    
    # Display the tables
    print("Effectiveness Table:")
    display(effectiveness_table)
    
    print("\nAdverse Events Table:")
    display(adverse_events_table)
    
    # Save to CSV for reference
    effectiveness_table.to_csv(CSV_DIR / "effectiveness_summary_table.csv", index=False)
    adverse_events_table.to_csv(CSV_DIR / "adverse_events_summary_table.csv", index=False)
    print("✓ Saved split summary tables to CSV")
    
    return effectiveness_table, adverse_events_table

# Run the function
effectiveness_table, adverse_events_table = create_and_save_split_tables()

Effectiveness Table:


,Platform,Medicine,N_All,Effectiveness_All,N_HQ,Effectiveness_HQ
5,Amazon,Chanca piedra,1193,69.8% (67.1-72.3),158,95.0% (90.4-97.5)
8,Amazon,Phosfood,40,70.0% (54.6-81.9),1,100.0% (20.7-100.0)
6,Amazon,Potassium citrate,133,55.6% (47.2-63.8),26,69.2% (50.0-83.5)
7,Amazon,Rowatinex,90,87.8% (79.4-93.0),20,100.0% (83.9-100.0)
12,Reddit,Allopurinol,49,14.3% (7.1-26.7),8,25.0% (7.1-59.1)
16,Reddit,Black seed,17,11.8% (3.3-34.3),3,33.3% (6.1-79.2)
11,Reddit,Chanca piedra,492,27.6% (23.9-31.8),75,58.7% (47.4-69.1)
9,Reddit,Flomax,1134,11.2% (9.5-13.2),232,18.5% (14.1-24.0)
14,Reddit,Garcinia,38,31.6% (19.1-47.5),1,0.0%
13,Reddit,Hydrochlorothiazide,46,13.0% (6.1-25.7),6,0.0%



Adverse Events Table:


,Platform,Medicine,N_All,Adverse_Events_All,N_HQ,Adverse_Events_HQ
9,Amazon,Chanca piedra,1193,2.7% (1.9-3.7),158,1.6% (0.5-5.0)
12,Amazon,Phosfood,40,7.5% (2.6-19.9),1,0.0%
10,Amazon,Potassium citrate,133,2.3% (0.8-6.4),26,3.8% (0.7-18.9)
11,Amazon,Rowatinex,90,2.2% (0.6-7.7),20,10.0% (2.8-30.1)
16,Reddit,Allopurinol,49,12.2% (5.7-24.2),8,12.5% (2.2-47.1)
20,Reddit,Black seed,17,0.0%,3,0.0%
15,Reddit,Chanca piedra,492,6.5% (4.6-9.0),75,14.7% (8.4-24.4)
13,Reddit,Flomax,1134,21.9% (19.6-24.4),232,25.0% (19.9-30.9)
18,Reddit,Garcinia,38,13.2% (5.8-27.3),1,100.0% (20.7-100.0)
17,Reddit,Hydrochlorothiazide,46,8.7% (3.4-20.3),6,16.7% (3.0-56.4)


✓ Saved split summary tables to CSV


In [ ]:

def create_effectiveness_table():
    """
    Create a table focused on effectiveness metrics across platforms:
    - Platform (WebMD, Amazon, Reddit)
    - Medicine name
    - Sample sizes (All / HQ)
    - Effectiveness percentages (All / HQ)
    """
    data = load_kidney_stone_data()
    
    # Products to exclude from effectiveness table (only for WebMD platform)
    webmd_excluded_products = ["Garcinia", "Black seed", "Ashwagandha", "Melatonin"]
    
    # Prepare an empty list to store effectiveness records
    effectiveness_summaries = []
    
    # Process each platform
    for platform, df in data.items():
        # Get all products with at least 5 reviews
        all_products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= 5].index.tolist()
        
        # Apply exclusion only for WebMD platform
        if platform == "WebMD":
            all_products = [product for product in all_products if product not in webmd_excluded_products]
        
        for product in all_products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Initialize variables
            helped_pct_all = 0
            helped_ci_low_all = float('nan')
            helped_ci_high_all = float('nan')
            n_hq = 0
            helped_pct_hq = float('nan')
            helped_ci_low_hq = float('nan')
            helped_ci_high_hq = float('nan')
            
            # Calculate effectiveness using weights
            helped_mask = all_df[HELP_COL].eq(1).fillna(False)
            weighted_helped = all_df.loc[helped_mask, 'Weight'].sum() if any(helped_mask) else 0
            total_weight = all_df['Weight'].sum()
            helped_pct_all = 100 * weighted_helped / total_weight if total_weight > 0 else 0
            
            # Add Wilson CIs
            if n_all > 0:
                # Effective counts for CI calculation
                effective_count = weighted_helped / total_weight * n_all if total_weight > 0 else 0
                if effective_count > 0:
                    helped_ci_low_all, helped_ci_high_all = proportion_confint(
                        effective_count, n_all, method="wilson"
                    )
                    helped_ci_low_all *= 100
                    helped_ci_high_all *= 100
            
            # HIGH QUALITY reviews (if applicable)
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq > 0:
                    # Calculate effectiveness for HQ
                    helped_mask_hq = hq_df[HELP_COL].eq(1).fillna(False)
                    helped_weighted_hq = hq_df.loc[helped_mask_hq, 'Weight'].sum() if any(helped_mask_hq) else 0
                    total_weight_hq = hq_df['Weight'].sum()
                    helped_pct_hq = 100 * helped_weighted_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # For CI calculation, use effective counts
                    if n_hq > 0:
                        effective_helped_hq = helped_weighted_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_helped_hq > 0:
                            helped_ci_low_hq, helped_ci_high_hq = proportion_confint(
                                effective_helped_hq, n_hq, method="wilson"
                            )
                            helped_ci_low_hq *= 100
                            helped_ci_high_hq *= 100
            
            # Add record to summaries
            effectiveness_summaries.append({
                "Platform": platform,
                "Medicine": product,
                "N_All": n_all,
                "N_HQ": n_hq,
                "Helped_Pct_All": helped_pct_all,
                "Helped_CI_Low_All": helped_ci_low_all,
                "Helped_CI_High_All": helped_ci_high_all,
                "Helped_Pct_HQ": helped_pct_hq,
                "Helped_CI_Low_HQ": helped_ci_low_hq,
                "Helped_CI_High_HQ": helped_ci_high_hq
            })
    
    # Convert to DataFrame and format
    effectiveness_df = pd.DataFrame(effectiveness_summaries)
    
    # Create a more readable format for effectiveness with CIs
    effectiveness_df["Effectiveness_All"] = effectiveness_df.apply(
        lambda x: f"{x['Helped_Pct_All']:.1f}% ({x['Helped_CI_Low_All']:.1f}-{x['Helped_CI_High_All']:.1f})" 
        if not pd.isna(x['Helped_CI_Low_All']) else f"{x['Helped_Pct_All']:.1f}%", 
        axis=1
    )
    
    effectiveness_df["Effectiveness_HQ"] = effectiveness_df.apply(
        lambda x: f"{x['Helped_Pct_HQ']:.1f}% ({x['Helped_CI_Low_HQ']:.1f}-{x['Helped_CI_High_HQ']:.1f})" 
        if not pd.isna(x['Helped_CI_Low_HQ']) else f"{x['Helped_Pct_HQ']:.1f}%" if not pd.isna(x['Helped_Pct_HQ']) else "N/A", 
        axis=1
    )
    
    # Select and order columns for final display
    display_cols = [
        "Platform", "Medicine", 
        "N_All", "Effectiveness_All",
        "N_HQ", "Effectiveness_HQ"
    ]
    
    # Sort by platform and then by medicine
    effectiveness_df = effectiveness_df.sort_values(["Platform", "Medicine"])
    
    return effectiveness_df[display_cols]

def create_adverse_events_table():
    """
    Create a table focused on adverse events metrics across platforms:
    - Platform (WebMD, Amazon, Reddit)
    - Medicine name
    - Sample sizes (All / HQ)
    - Adverse Events percentages (All / HQ)
    """
    data = load_kidney_stone_data()
    
    # Prepare an empty list to store adverse events records
    ae_summaries = []
    
    # Process each platform
    for platform, df in data.items():
        # Get all products with at least 5 reviews
        all_products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= 5].index.tolist()
        
        for product in all_products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Initialize variables
            ae_pct_all = 0
            ae_ci_low_all = float('nan')
            ae_ci_high_all = float('nan')
            n_hq = 0
            ae_pct_hq = float('nan')
            ae_ci_low_hq = float('nan')
            ae_ci_high_hq = float('nan')
            
            # Calculate adverse events
            ae_mask = all_df[AE_COL].eq(1).fillna(False)
            ae_weighted = all_df.loc[ae_mask, 'Weight'].sum() if any(ae_mask) else 0
            total_weight = all_df['Weight'].sum()
            
            # Calculate percentage using weights
            ae_pct_all = 100 * ae_weighted / total_weight if total_weight > 0 else 0
            
            # For CI calculation, use effective counts
            if n_all > 0:
                effective_ae = ae_weighted / total_weight * n_all if total_weight > 0 else 0
                if effective_ae > 0:
                    ae_ci_low_all, ae_ci_high_all = proportion_confint(
                        effective_ae, n_all, method="wilson"
                    )
                    ae_ci_low_all *= 100
                    ae_ci_high_all *= 100
            
            # HIGH QUALITY reviews (if applicable)
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq > 0:
                    # Calculate adverse events for HQ
                    ae_mask_hq = hq_df[AE_COL].eq(1).fillna(False)
                    ae_weighted_hq = hq_df.loc[ae_mask_hq, 'Weight'].sum() if any(ae_mask_hq) else 0
                    total_weight_hq = hq_df['Weight'].sum()
                    ae_pct_hq = 100 * ae_weighted_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # For CI calculation, use effective counts
                    if n_hq > 0:
                        effective_ae_hq = ae_weighted_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_ae_hq > 0:
                            ae_ci_low_hq, ae_ci_high_hq = proportion_confint(
                                effective_ae_hq, n_hq, method="wilson"
                            )
                            ae_ci_low_hq *= 100
                            ae_ci_high_hq *= 100
            
            # Add record to summaries
            ae_summaries.append({
                "Platform": platform,
                "Medicine": product,
                "N_All": n_all,
                "N_HQ": n_hq,
                "AE_Pct_All": ae_pct_all,
                "AE_CI_Low_All": ae_ci_low_all,
                "AE_CI_High_All": ae_ci_high_all,
                "AE_Pct_HQ": ae_pct_hq,
                "AE_CI_Low_HQ": ae_ci_low_hq,
                "AE_CI_High_HQ": ae_ci_high_hq
            })
    
    # Convert to DataFrame and format
    ae_df = pd.DataFrame(ae_summaries)
    
    # Create a more readable format for AE with CIs
    ae_df["Adverse_Events_All"] = ae_df.apply(
        lambda x: f"{x['AE_Pct_All']:.1f}% ({x['AE_CI_Low_All']:.1f}-{x['AE_CI_High_All']:.1f})" 
        if not pd.isna(x['AE_CI_Low_All']) else f"{x['AE_Pct_All']:.1f}%", 
        axis=1
    )
    
    ae_df["Adverse_Events_HQ"] = ae_df.apply(
        lambda x: f"{x['AE_Pct_HQ']:.1f}% ({x['AE_CI_Low_HQ']:.1f}-{x['AE_CI_High_HQ']:.1f})" 
        if not pd.isna(x['AE_CI_Low_HQ']) else f"{x['AE_Pct_HQ']:.1f}%" if not pd.isna(x['AE_Pct_HQ']) else "N/A", 
        axis=1
    )
    
    # Select and order columns for final display
    display_cols = [
        "Platform", "Medicine", 
        "N_All", "Adverse_Events_All",
        "N_HQ", "Adverse_Events_HQ"
    ]
    
    # Sort by platform and then by medicine
    ae_df = ae_df.sort_values(["Platform", "Medicine"])
    
    return ae_df[display_cols]

# Function to run both tables and save them
def create_and_save_split_tables():
    """
    Create both effectiveness and adverse events tables and save them to CSV
    """
    # Generate the tables
    effectiveness_table = create_effectiveness_table()
    adverse_events_table = create_adverse_events_table()
    
    # Display the tables
    print("Effectiveness Table:")
    display(effectiveness_table)
    
    print("\nAdverse Events Table:")
    display(adverse_events_table)
    
    # Save to CSV for reference
    effectiveness_table.to_csv(CSV_DIR / "effectiveness_summary_table.csv", index=False)
    adverse_events_table.to_csv(CSV_DIR / "adverse_events_summary_table.csv", index=False)
    print("✓ Saved split summary tables to CSV")
    
    return effectiveness_table, adverse_events_table

# Run the function
effectiveness_table, adverse_events_table = create_and_save_split_tables()

## 2. Multi-Panel Forest Plots 
Two visualizations:
1. A multi-panel forest plot showing effectiveness across platforms
2. A multi-panel forest plot showing adverse events across platforms

In [36]:
def create_multipanel_forest_plots():
    data = load_kidney_stone_data()
    
    # Prepare data for plotting
    effectiveness_data = []
    adverse_events_data = []
    exclude_eff = {"Melatonin", "Ashwagandha"}

    for platform, df in data.items():
        products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= 1].index.tolist()
        
        for product in products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Calculate effectiveness using weights
            helped_mask = all_df[HELP_COL].eq(1).fillna(False)
            weighted_helped = all_df.loc[helped_mask, 'Weight'].sum()
            total_weight = all_df['Weight'].sum()
            helped_pct_all = 100 * weighted_helped / total_weight if total_weight > 0 else 0
            
            # Calculate CI
            if n_all > 0:
                effective_count = weighted_helped / total_weight * n_all if total_weight > 0 else 0
                if effective_count > 0:
                    helped_ci_low_all, helped_ci_high_all = proportion_confint(
                        effective_count, n_all, method="wilson"
                    )
                    helped_ci_low_all *= 100
                    helped_ci_high_all *= 100
                else:
                    helped_ci_low_all = helped_ci_high_all = np.nan
            else:
                helped_ci_low_all = helped_ci_high_all = np.nan
            
            # Calculate adverse events using weights
            ae_mask = all_df[AE_COL].eq(1).fillna(False)
            weighted_ae = all_df.loc[ae_mask, 'Weight'].sum()
            ae_pct_all = 100 * weighted_ae / total_weight if total_weight > 0 else 0
            
            # Calculate CI
            if n_all > 0:
                effective_count = weighted_ae / total_weight * n_all if total_weight > 0 else 0
                if effective_count > 0:
                    ae_ci_low_all, ae_ci_high_all = proportion_confint(
                        effective_count, n_all, method="wilson"
                    )
                    ae_ci_low_all *= 100
                    ae_ci_high_all *= 100
                else:
                    ae_ci_low_all = ae_ci_high_all = np.nan
            else:
                ae_ci_low_all = ae_ci_high_all = np.nan
            
            # Add to data
            if product not in exclude_eff:
                effectiveness_data.append({
                    "Platform": platform,
                    "Medicine": product,
                    "Type": "All Reviews",
                    "Percentage": helped_pct_all,
                    "CI_Low": helped_ci_low_all,
                    "CI_High": helped_ci_high_all,
                    "N": n_all
                })
            
            adverse_events_data.append({
                "Platform": platform,
                "Medicine": product,
                "Type": "All Reviews",
                "Percentage": ae_pct_all,
                "CI_Low": ae_ci_low_all,
                "CI_High": ae_ci_high_all,
                "N": n_all
            })
            
            # HIGH QUALITY reviews
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq >= 1:
                    # Calculate effectiveness using weights
                    helped_mask_hq = hq_df[HELP_COL].eq(1).fillna(False)
                    weighted_helped_hq = hq_df.loc[helped_mask_hq, 'Weight'].sum()
                    total_weight_hq = hq_df['Weight'].sum()
                    helped_pct_hq = 100 * weighted_helped_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # Calculate CI
                    if n_hq > 0:
                        effective_count = weighted_helped_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_count > 0:
                            helped_ci_low_hq, helped_ci_high_hq = proportion_confint(
                                effective_count, n_hq, method="wilson"
                            )
                            helped_ci_low_hq *= 100
                            helped_ci_high_hq *= 100
                        else:
                            helped_ci_low_hq = helped_ci_high_hq = np.nan
                    else:
                        helped_ci_low_hq = helped_ci_high_hq = np.nan
                    
                    # Calculate adverse events using weights
                    ae_mask_hq = hq_df[AE_COL].eq(1).fillna(False)
                    weighted_ae_hq = hq_df.loc[ae_mask_hq, 'Weight'].sum()
                    ae_pct_hq = 100 * weighted_ae_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # Calculate CI
                    if n_hq > 0:
                        effective_count = weighted_ae_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_count > 0:
                            ae_ci_low_hq, ae_ci_high_hq = proportion_confint(
                                effective_count, n_hq, method="wilson"
                            )
                            ae_ci_low_hq *= 100
                            ae_ci_high_hq *= 100
                        else:
                            ae_ci_low_hq = ae_ci_high_hq = np.nan
                    else:
                        ae_ci_low_hq = ae_ci_high_hq = np.nan
                    
                    # Add to data
                    if product not in exclude_eff:
                        effectiveness_data.append({
                            "Platform": platform,
                            "Medicine": product,
                            "Type": "High Quality",
                            "Percentage": helped_pct_hq,
                            "CI_Low": helped_ci_low_hq,
                            "CI_High": helped_ci_high_hq,
                            "N": n_hq
                        })
                    
                    adverse_events_data.append({
                        "Platform": platform,
                        "Medicine": product,
                        "Type": "High Quality",
                        "Percentage": ae_pct_hq,
                        "CI_Low": ae_ci_low_hq,
                        "CI_High": ae_ci_high_hq,
                        "N": n_hq
                    })
    
    # Convert to DataFrames
    effectiveness_df = pd.DataFrame(effectiveness_data)
    adverse_events_df = pd.DataFrame(adverse_events_data)
    
    # Create plots
    create_panel_forest_plot(effectiveness_df, "Effectiveness", "% Reporting Help")
    create_panel_forest_plot(adverse_events_df, "Adverse Events", "% Reporting Adverse Events")

def create_panel_forest_plot(data_df, plot_title, y_axis_title):
    """
    Create a multi-panel forest plot showing results across platforms.
    
    Parameters:
    - data_df: DataFrame with columns Platform, Medicine, Type, Percentage, CI_Low, CI_High, N
    - plot_title: Title for the overall plot
    - y_axis_title: Label for the y-axis
    """
    # Set platform order and colors
    platform_order = ["WebMD", "Amazon", "Reddit"]
    platform_colors = {"WebMD": "#636EFA", "Amazon": "#EF553B", "Reddit": "#00CC96"}
    
    # Create multi-panel figure
    from plotly.subplots import make_subplots
    fig = make_subplots(rows=1, cols=3, 
                        subplot_titles=platform_order,
                        shared_yaxes=True,
                        horizontal_spacing=0.05)  # Increased spacing
    
    # Get the unique medications across all platforms for consistent y-axis
    all_meds = data_df["Medicine"].unique()
    all_meds = sorted(all_meds, reverse=True)
    if plot_title == "Effectiveness":
        webmd_exclude = ["Black seed", "Garcinia"]
        data_df = data_df[~((data_df["Platform"] == "WebMD") & 
                            (data_df["Medicine"].isin(webmd_exclude)))]
    
    # Create a mapping of medicine names to numeric positions
    med_to_pos = {med: i for i, med in enumerate(sorted(all_meds, reverse=True))}
    
    # Define y-offset for separation between All Reviews and High Quality points
    y_offset = 0.20  # Adjust as needed for visual separation
    
    # Process each platform
    for i, platform in enumerate(platform_order):
        # Filter data for this platform
        platform_data = data_df[data_df["Platform"] == platform]
        
        if len(platform_data) == 0:
            continue
        
        # Add All Reviews trace
        all_data = platform_data[platform_data["Type"] == "All Reviews"]
        all_data = all_data[all_data["N"] >= 10]
        all_data["CI_Low_Display"] = all_data["CI_Low"].round(1)
        all_data["CI_High_Display"] = all_data["CI_High"].round(1)
        
        # Calculate y positions with offset for All Reviews (shift up)
        all_data["y_position"] = all_data["Medicine"].apply(lambda x: med_to_pos[x])
        
        # Add trace for All Reviews - Data points with error bars
        if len(all_data) > 0:
            fig.add_trace(
                go.Scatter(
                    x=all_data["Percentage"],
                    y=all_data["y_position"],  # Use numeric position with offset
                    error_x=dict(
                        type='data',
                        symmetric=False,
                        array=all_data["CI_High"] - all_data["Percentage"],
                        arrayminus=all_data["Percentage"] - all_data["CI_Low"],
                        color=platform_colors[platform],
                    ),
                    mode="markers",
                    marker=dict(
                        symbol="circle",
                        size=10,
                        color=platform_colors[platform],
                        opacity=1.0,
                        line=dict(width=2, color="DarkSlateGrey")
                    ),
                    name=f"{platform} - All Reviews",
                    text=[f"{med}<br>N={n}" for med, n in zip(all_data["Medicine"], all_data["N"])],
                    hovertemplate="%{text}<br>Value: %{x:.1f}% (%{customdata[0]}-%{customdata[1]})",
                    showlegend=i==0,
                    customdata=all_data[["CI_Low_Display", "CI_High_Display"]].values
                ),
                row=1, col=i+1
            )
            
            # Add sample size labels above the All Reviews points
            for j, row in all_data.iterrows():
                fig.add_trace(
                    go.Scatter(
                        x=[row["Percentage"] + 5 if row["Percentage"] <= 5 else (row["Percentage"] - 5 if row["Percentage"] >= 95 else row["Percentage"])],
                        y=[row["y_position"] + 0.1],  # Slightly above the point
                        mode="text",
                        text=[f"n={row['N']}"],
                        textposition="top center",
                        textfont=dict(
                            color=platform_colors[platform],
                            size=9
                        ),
                        showlegend=False,
                        hoverinfo="skip"
                    ),
                    row=1, col=i+1
                )
        
        # Add High Quality trace if present
        hq_data = platform_data[platform_data["Type"] == "High Quality"]
        hq_data = hq_data[hq_data["N"] >= 10]
        hq_data["CI_Low_Display"] = hq_data["CI_Low"].round(1)
        hq_data["CI_High_Display"] = hq_data["CI_High"].round(1)

        # Calculate y positions with offset for High Quality Reviews (shift down)
        hq_data["y_position"] = hq_data["Medicine"].apply(lambda x: med_to_pos[x] - y_offset)
        
        if len(hq_data) > 0:
            # Add High Quality points with error bars
            fig.add_trace(
                go.Scatter(
                    x=hq_data["Percentage"],
                    y=hq_data["y_position"],  # Use numeric position with offset
                    error_x=dict(
                        type='data',
                        symmetric=False,
                        array=hq_data["CI_High"] - hq_data["Percentage"],
                        arrayminus=hq_data["Percentage"] - hq_data["CI_Low"],
                        color=platform_colors[platform],
                    ),
                    mode="markers",
                    marker=dict(
                        symbol="diamond",  # Different shape
                        size=10,
                        color=platform_colors[platform],
                        opacity=0.6,  # Transparency
                        line=dict(width=2, color="black")
                    ),
                    name=f"{platform} - High Quality",
                    text=[f"{med}<br>N={n}" for med, n in zip(hq_data["Medicine"], hq_data["N"])],
                    hovertemplate="%{text}<br>Value: %{x:.1f}% (%{customdata[0]}-%{customdata[1]})",
                    showlegend=i==0,
                    customdata=hq_data[["CI_Low_Display", "CI_High_Display"]].values
                ),
                row=1, col=i+1
            )
            
            # Add sample size labels below the High Quality points
            for j, row in hq_data.iterrows():
                fig.add_trace(
                    go.Scatter(
                        x=[row["Percentage"] + 5 if row["Percentage"] <= 5 else (row["Percentage"] - 5 if row["Percentage"] >= 95 else row["Percentage"])],
                        y=[row["y_position"] - y_offset],  # Slightly below the point
                        mode="text",
                        text=[f"n={row['N']}"],
                        textposition="bottom center",
                        textfont=dict(
                            color=platform_colors[platform],
                            size=9
                        ),
                        showlegend=False,
                        hoverinfo="skip"
                    ),
                    row=1, col=i+1
                )
    
    # Update layout
    fig.update_layout(
        title=plot_title,
        height=max(600, 100 + 40 * len(all_meds)),  # Dynamic height based on number of medicines
        width=1000,  # Original width
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # Create tick positions at the exact positions of the medicine names
    tick_positions = list(med_to_pos.values())
    tick_labels = list(med_to_pos.keys())
    
    # Update x-axes and y-axes for each subplot
    for j in range(3):
        fig.update_xaxes(
            title=y_axis_title,
            range=[0, 100],  # Keep original range
            tickvals=[0, 20, 40, 60, 80, 100],
            ticktext=["0", "20", "40", "60", "80", "100"],
            tickfont=dict(size=10),
            row=1, col=j+1
        )
        
        # Set custom tick positions and labels
        fig.update_yaxes(
            tickmode='array',
            tickvals=tick_positions,
            ticktext=tick_labels,
            row=1, col=j+1
        )
    
    # Show the plot
    fig.show()
    
    # Save as HTML
    output_path = CSV_DIR / f"forest_plot_{plot_title.lower().replace(' ', '_')}_with_labels.html"
    fig.write_html(output_path)
    print(f"✓ Saved forest plot to {output_path}")
    
    # Try to save as PNG if kaleido is installed
    try:
        png_path = CSV_DIR / f"forest_plot_{plot_title.lower().replace(' ', '_')}_with_labels.png"
        # fig.write_image(png_path, scale=2)
        print(f"✓ Saved PNG version to {png_path}")
    except Exception as e:
        print(f"Note: Could not save PNG (install 'kaleido' package if needed). Error: {e}")
    
    return fig

# Run the function to create the forest plots
multipanel_plots = create_multipanel_forest_plots()

✓ Saved forest plot to csv-files/forest_plot_effectiveness_with_labels.html
✓ Saved PNG version to csv-files/forest_plot_effectiveness_with_labels.png


✓ Saved forest plot to csv-files/forest_plot_adverse_events_with_labels.html
✓ Saved PNG version to csv-files/forest_plot_adverse_events_with_labels.png


## 3. Regression tests

We summarize the results in four tables (Helped/Adverse × All/HQ)

In [37]:
# ---------------------------------------------------------------------
# Fisher exact (unweighted) – OR, CI, p
# ---------------------------------------------------------------------

def fisher_stats(tbl: pd.DataFrame):
    """
    Exact OR, exact CI (mid-p), and Fisher two-sided p-value
    for a 2×2 contingency table.
    """
    table = Table2x2(tbl.to_numpy())

    # point estimate and exact conditional CI
    or_hat = table.oddsratio
    ci_low, ci_high = table.oddsratio_confint(method="exact")

    # Fisher p-value via SciPy
    _, p_val = fisher_exact(tbl.to_numpy())      # two-sided by default

    return or_hat, (ci_low, ci_high), p_val

# ---------------------------------------------------------------------
# Logistic regression (weighted, robust SEs)
# ---------------------------------------------------------------------

def glm_stats(df: pd.DataFrame, ref: str, outcome: str) -> Tuple[float, Tuple[float, float], float]:
    df = df.assign(y=df[outcome].eq(1).fillna(False).astype(int)) 
    cats = [ref] + [c for c in df["Medicine"].unique() if c != ref]
    df["Medicine"] = pd.Categorical(df["Medicine"], categories=cats)

    model = glm(
        "y ~ C(Medicine)",
        data=df,
        family=sm.families.Binomial(),
        freq_weights=df["Weight"],
    ).fit(cov_type="HC0")

    param = f"C(Medicine)[T.{cats[1]}]"  # first comparator only
    or_hat = math.exp(model.params[param])
    ci_low, ci_high = (math.exp(v) for v in model.conf_int().loc[param])
    return or_hat, (ci_low, ci_high), model.pvalues[param]

# ---------------------------------------------------------------------
# Single comparator wrapper
# ---------------------------------------------------------------------

def one_result(df: pd.DataFrame, ref: str, comp: str, col: str):
    tbl = _2x2(df, ref, comp, col)
    method = choose_method(tbl)

    if method == "describe":
        return {"Method": "Describe", "OR": None, "CI": (None, None), "p": None}
    if method == "fisher":
        or_hat, ci, p = fisher_stats(tbl)
        return {"Method": "Fisher", "OR": or_hat, "CI": ci, "p": p}
    or_hat, ci, p = glm_stats(df[df["Medicine"].isin([ref, comp])], ref, col)
    return {"Method": "Logit", "OR": or_hat, "CI": ci, "p": p}

# ---------------------------------------------------------------------
# Flat OR report for all comparators
# ---------------------------------------------------------------------

def odds_ratio_report(ref_product: str = "Chanca piedra", *, hq_only: bool = False) -> pd.DataFrame:
    rows: List[dict] = []
    for platform, df in load_kidney_stone_data().items():
        if ref_product not in df["Medicine"].unique():
            continue
        if hq_only and HQ_COL in df.columns:
            df = df[df[HQ_COL] == 1]
        for comp in set(df["Medicine"]) - {ref_product}:
            for col, outcome in [(HELP_COL, "Helped"), (AE_COL, "Adverse")]:
                res = one_result(df, ref_product, comp, col)
                rows.append({
                    "Platform": platform,
                    "Comparator": comp,
                    "Outcome": outcome,
                    "Method": res["Method"],
                    "OR": res["OR"],
                    "CI_low": res["CI"][0],
                    "CI_high": res["CI"][1],
                    "p": res["p"],
                })
    return pd.DataFrame(rows)

In [38]:
# ---------------------------------------------------------------------
# Four summary tables (Helped/Adverse × All/HQ)
# ---------------------------------------------------------------------

def summary_tables(ref_product: str = "Chanca piedra", *, min_include_n: int = 5):
    data = load_kidney_stone_data()

    # counts dict: {(platform, medicine): {"N_All": int, "N_HQ": int}}
    counts: Dict[Tuple[str, str], dict] = {}
    for platform, df in data.items():
        for med, grp in df.groupby("Medicine"):
            counts[(platform, med)] = {
                "N_All": len(grp),
                "N_HQ": len(grp[grp[HQ_COL] == 1]) if HQ_COL in df.columns else 0,
            }

    # helper ─ attach N_Ref / N_Comp counts to an OR dataframe
    def _attach(df_or: pd.DataFrame, which: str) -> pd.DataFrame:
        """Add count columns to a platform-comparator OR table."""
        df = df_or.copy()

        # reference-arm counts are always present
        df["N_Ref"] = df["Platform"].map(lambda p: counts[(p, ref_product)][which])

        # safe lookup for comparator counts (may be zero after filtering)
        def _comp_count(row):
            key = (row["Platform"], row["Comparator"])
            return counts.get(key, {which: 0})[which]

        df["N_Comp"] = df.apply(_comp_count, axis=1)
        return df


    # generate OR reports -----------------------------------------------------
    or_all = odds_ratio_report(ref_product, hq_only=False)
    or_hq = odds_ratio_report(ref_product, hq_only=True)

    tables = {}
    tables["helped_all"] = _attach(or_all[or_all["Outcome"] == "Helped"], "N_All")
    tables["helped_hq"] = _attach(or_hq[or_hq["Outcome"] == "Helped"], "N_HQ")
    tables["ae_all"] = _attach(or_all[or_all["Outcome"] == "Adverse"], "N_All")
    tables["ae_hq"] = _attach(or_hq[or_hq["Outcome"] == "Adverse"], "N_HQ")

    # optional min_include_n filter ------------------------------------------
    for key, t in tables.items():
        tables[key] = t[(t["N_Ref"] >= min_include_n) & (t["N_Comp"] >= min_include_n)].reset_index(drop=True)
    return (
        tables["helped_all"],
        tables["helped_hq"],
        tables["ae_all"],
        tables["ae_hq"],
    )

def _star(p):
    if p < 0.001:
        return " ***"
    if p < 0.01:
        return " **"
    if p < 0.05:
        return " *"
    return ""

def _fmt_or(or_val, lo, hi, p):
    """Return 'OR (low-high)' with 2-decimals (or 3 if very small)."""
    if pd.isna(or_val):
        return "N/A"
    small = 0 < min(or_val, lo, hi) < 0.01
    digits = 3 if small else 2
    s = f"{or_val:.{digits}f} ({lo:.{digits}f}–{hi:.{digits}f})"
    return s + _star(p)

def beautify(df, kind, *, hq=False):
    df_k = df[df["Outcome"] == kind].copy()

    df_k["OR_pretty"] = df_k.apply(
        lambda r: _fmt_or(r["OR"], r["CI_low"], r["CI_high"], r["p"]),
        axis=1,
    )

    suffix = "HQ" if hq else "All"

    return (
        df_k[["Platform", "Comparator", "OR_pretty", "Method",
              "N_Ref", "N_Comp"]]
        .rename(columns={
            "Comparator": "Comparison",
            "Method": f"Test_Method_{kind}_{suffix}",
            "OR_pretty": f"OR_{kind}_{suffix}",
        })
        .sort_values(["Platform", "Comparison"])
        .reset_index(drop=True)
    )

h_all, h_hq, ae_all, ae_hq = summary_tables()

helped_all_pretty = beautify(h_all, "Helped", hq=False)
helped_hq_pretty  = beautify(h_hq, "Helped", hq=True)
ae_all_pretty     = beautify(ae_all, "Adverse", hq=False)
ae_hq_pretty      = beautify(ae_hq, "Adverse", hq=True)

print("HELPED – ALL REVIEWS")
display(helped_all_pretty)

print("\nHELPED – HIGH QUALITY")
display(helped_hq_pretty)

print("\nADVERSE – ALL REVIEWS")
display(ae_all_pretty)

print("\nADVERSE – HIGH QUALITY")
display(ae_hq_pretty)

# Save all four tables to CSV
helped_all_pretty.to_csv(CSV_DIR / "chanca_piedra_helped_all_table.csv", index=False)
helped_hq_pretty.to_csv(CSV_DIR / "chanca_piedra_helped_hq_table.csv", index=False)
ae_all.to_csv(CSV_DIR / "chanca_piedra_ae_all_table.csv", index=False)
ae_hq_pretty.to_csv(CSV_DIR / "chanca_piedra_ae_hq_table.csv", index=False)

HELPED – ALL REVIEWS


,Platform,Comparison,OR_Helped_All,Test_Method_Helped_All,N_Ref,N_Comp
0,Amazon,Phosfood,1.01 (0.51–2.00),Logit,1193,40
1,Amazon,Potassium citrate,0.54 (0.38–0.77) ***,Logit,1193,133
2,Amazon,Rowatinex,3.11 (1.65–5.88) ***,Logit,1193,90
3,Reddit,Allopurinol,0.44 (0.19–0.99) *,Fisher,492,49
4,Reddit,Black seed,N/A,Describe,492,17
5,Reddit,Flomax,0.33 (0.25–0.43) ***,Logit,492,1134
6,Reddit,Garcinia,1.21 (0.59–2.46),Logit,492,38
7,Reddit,Hydrochlorothiazide,0.39 (0.16–0.95) *,Fisher,492,46
8,Reddit,Potassium citrate,0.42 (0.30–0.58) ***,Logit,492,509
9,Reddit,Rowatinex,0.82 (0.29–2.28),Fisher,492,21



HELPED – HIGH QUALITY


,Platform,Comparison,OR_Helped_HQ,Test_Method_Helped_HQ,N_Ref,N_Comp
0,Amazon,Potassium citrate,0.33 (0.13–0.85) *,Fisher,158,26
1,Amazon,Rowatinex,N/A,Describe,158,20
2,Reddit,Allopurinol,N/A,Describe,75,8
3,Reddit,Flomax,0.16 (0.09–0.28) ***,Logit,75,232
4,Reddit,Hydrochlorothiazide,N/A,Describe,75,6
5,Reddit,Potassium citrate,0.19 (0.10–0.35) ***,Logit,75,147
6,WebMD,Ashwagandha,N/A,Describe,33,48
7,WebMD,Black seed,N/A,Describe,33,14
8,WebMD,Flomax,N/A,Describe,33,5
9,WebMD,Garcinia,N/A,Describe,33,71



ADVERSE – ALL REVIEWS


,Platform,Comparison,OR_Adverse_All,Test_Method_Adverse_All,N_Ref,N_Comp
0,Amazon,Phosfood,N/A,Describe,1193,40
1,Amazon,Potassium citrate,N/A,Describe,1193,133
2,Amazon,Rowatinex,N/A,Describe,1193,90
3,Reddit,Allopurinol,2.01 (0.79–5.06),Fisher,492,49
4,Reddit,Black seed,N/A,Describe,492,17
5,Reddit,Flomax,4.02 (2.74–5.91) ***,Logit,492,1134
6,Reddit,Garcinia,2.18 (0.80–5.96),Fisher,492,38
7,Reddit,Hydrochlorothiazide,N/A,Describe,492,46
8,Reddit,Potassium citrate,2.22 (1.43–3.44) ***,Logit,492,509
9,Reddit,Rowatinex,N/A,Describe,492,21



ADVERSE – HIGH QUALITY


,Platform,Comparison,OR_Adverse_HQ,Test_Method_Adverse_HQ,N_Ref,N_Comp
0,Amazon,Potassium citrate,N/A,Describe,158,26
1,Amazon,Rowatinex,N/A,Describe,158,20
2,Reddit,Allopurinol,N/A,Describe,75,8
3,Reddit,Flomax,1.94 (0.96–3.93),Logit,75,232
4,Reddit,Hydrochlorothiazide,N/A,Describe,75,6
5,Reddit,Potassium citrate,0.97 (0.44–2.13),Logit,75,147
6,WebMD,Ashwagandha,N/A,Describe,33,48
7,WebMD,Black seed,N/A,Describe,33,14
8,WebMD,Flomax,N/A,Describe,33,5
9,WebMD,Garcinia,N/A,Describe,33,71
